In [4]:
import pandas as pd 
import numpy as np 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.svm import SVR

In [5]:
data = pd.read_csv('data/raw/train.csv')
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/train.csv'

In [ ]:

def year_features(df):
    df = df.copy()

    # 1. Age de la maison au moment de la vente
    df["House_Age"] = df["YrSold"] - df["YearBuilt"]

    # 2. Age depuis la dernière rénovation
    df["Remod_Age"] = df["YrSold"] - df["YearRemodAdd"]

    # 3. Indiquer si la maison est neuve
    df["IsNewHouse"] = (df["YrSold"] == df["YearBuilt"]).astype(int)

    # 4. Indiquer si la maison a été rénovée
    df["WasRemodeled"] = (
        df["YearRemodAdd"] != df["YearBuilt"]
    ).astype(int)

    # 5. Présence d'un garage
    df["Has_Garage"] = (
        df["GarageType"].notna()
        & (df["GarageType"] != "None")
    ).astype(int)

    # 6. Age du garage
    
    df["Garage_Age"] = df["YrSold"] - df["GarageYrBlt"]

    # Si la maison n'a pas de garage, on conserve NaN.
    df.loc[df["Has_Garage"] == 0, "Garage_Age"] = np.nan

    # 7. Encodage cyclique du mois de vente
    angle = 2 * np.pi * (df["MoSold"] - 1) / 12

    df["MoSold_sin"] = np.sin(angle)
    df["MoSold_cos"] = np.cos(angle)


    # 8. Supprimer les variables devenues inutiles
    df = df.drop(columns=[
        "YearBuilt",
        "YearRemodAdd",
        "GarageYrBlt",
        "MoSold"
    ])

    return df

In [ ]:
# faire une preprocessing prélimenaire 
def data_filter(data):
    data = data.copy()

    # 1. Traiter les absences réelles d'équipements
    colonnes_absence = [
        "Alley", "BsmtQual", "BsmtCond", "BsmtExposure",
        "BsmtFinType1", "BsmtFinType2", "FireplaceQu",
        "GarageType", "GarageFinish", "GarageQual",
        "GarageCond", "PoolQC", "Fence", "MiscFeature"
    ]

    cols_presentes = [
        c for c in colonnes_absence if c in data.columns
    ]

    data[cols_presentes] = (
        data[cols_presentes].fillna("None")
    )

    # 2. Créer les variables temporelles
    data = year_features(data)

    # 3. Transformer les variables asymétriques
    cols_log = [
        "MiscVal", "PoolArea", "LotArea", "3SsnPorch",
        "LowQualFinSF", "BsmtFinSF2", "ScreenPorch",
        "EnclosedPorch", "OpenPorchSF", "WoodDeckSF",
        "MasVnrArea", "LotFrontage", "BsmtFinSF1",
        "TotalBsmtSF", "1stFlrSF", "GrLivArea"
    ]

    cols_presentes = [
        c for c in cols_log if c in data.columns
    ]

    # Vérifier que les valeurs non manquantes sont >= 0
    for col in cols_presentes:
        valeurs = data[col].dropna()

        if (valeurs < 0).any():
            raise ValueError(
                f"Valeurs négatives détectées dans {col}"
            )

    data[cols_presentes] = np.log1p(
        data[cols_presentes]
    )

    # 4. MSSubClass est catégorielle
    if "MSSubClass" in data.columns:
        data["MSSubClass"] = (
            data["MSSubClass"].astype("string")
        )

    # 5. Cible en log, seulement si elle existe (absente du test Kaggle),
    #    puis placée à la fin
    if "SalePrice" in data.columns:
        if data["SalePrice"].max() > 100:             # garde-fou : pas de double log1p
            data["SalePrice"] = np.log1p(data["SalePrice"])
        data = data[
            [c for c in data.columns if c != "SalePrice"]
            + ["SalePrice"]
        ]

    return data

In [ ]:
data = data_filter(data)
data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,SaleCondition,House_Age,Remod_Age,IsNewHouse,WasRemodeled,Has_Garage,Garage_Age,MoSold_sin,MoSold_cos,SalePrice
0,1,60,RL,4.189655,9.042040,Pave,None,Reg,Lvl,AllPub,...,Normal,5,5,0,0,1,5.0,0.500000,0.866025,12.247699
1,2,20,RL,4.394449,9.169623,Pave,None,Reg,Lvl,AllPub,...,Normal,31,31,0,0,1,31.0,0.866025,-0.500000,12.109016
2,3,60,RL,4.234107,9.328212,Pave,None,IR1,Lvl,AllPub,...,Normal,7,6,0,1,1,7.0,-0.866025,-0.500000,12.317171
3,4,70,RL,4.110874,9.164401,Pave,None,IR1,Lvl,AllPub,...,Abnorml,91,36,0,1,1,8.0,0.500000,0.866025,11.849405
4,5,60,RL,4.442651,9.565284,Pave,None,IR1,Lvl,AllPub,...,Normal,8,8,0,0,1,8.0,-0.500000,0.866025,12.429220


In [ ]:
# application de cross validation train / test split 
data_train, data_test = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
data_train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,SaleCondition,House_Age,Remod_Age,IsNewHouse,WasRemodeled,Has_Garage,Garage_Age,MoSold_sin,MoSold_cos,SalePrice
254,255,20,RL,4.262680,9.036106,Pave,None,Reg,Lvl,AllPub,...,Normal,53,53,0,0,1,53.0,0.500000,-0.866025,11.884496
1066,1067,60,RL,4.094345,8.966739,Pave,None,IR1,Lvl,AllPub,...,Normal,16,15,0,1,1,16.0,0.866025,-0.500000,12.089544
638,639,30,RL,4.219508,9.080004,Pave,None,Reg,Lvl,AllPub,...,Normal,98,58,0,1,0,NaN,0.866025,-0.500000,11.350418
799,800,50,RL,4.110874,8.881975,Pave,None,Reg,Lvl,AllPub,...,Normal,70,57,0,1,1,68.0,0.500000,-0.866025,12.072547
380,381,50,RL,3.931826,8.517393,Pave,Pave,Reg,Lvl,AllPub,...,Normal,86,60,0,1,1,86.0,0.866025,-0.500000,11.751950


data prerocessing

In [ ]:
# fixer les parametre qu'on va utilisé 
MAX_GRLIVAREA = 4000
FACTEUR_IQR = 3   # 3 au lieu de 1,5 : sur des variables asymétriques, 1,5 en signale trop

# Colonnes numériques à ne jamais plafonner : identifiant, cible, codes de catégories
COLONNES_EXCLUES = ("Id", "SalePrice", "MSSubClass", "MoSold")


def retirer_grandes_maisons(train):
    """Retire les maisons avec GrLivArea > 4000 (à appliquer au TRAIN seulement).

    Si GrLivArea est déjà en log, le seuil est converti en log aussi.
    """
    if "GrLivArea" not in train.columns:
        return train

    en_log = train["GrLivArea"].max() < 100          # en vérifie si le variable est déja transformer en log1p ou non 
    seuil = np.log1p(MAX_GRLIVAREA) if en_log else MAX_GRLIVAREA # si en log1p en utilise seuil 100 si non en utilise 400

    train_propre = train[train["GrLivArea"] <= seuil]
    print(f"GrLivArea > {MAX_GRLIVAREA} : {len(train) - len(train_propre)} maison(s) retirée(s).")
    return train_propre


def apprendre_bornes_iqr(train, facteur=FACTEUR_IQR, colonnes_exclues=COLONNES_EXCLUES):
    """Calcule les bornes IQR de chaque variable numérique du TRAIN.

    Renvoie un dictionnaire {colonne: (borne_basse, borne_haute)}.
    Les colonnes dont l'IQR vaut 0 n'y figurent pas : on ne fait rien dessus.
    """
    bornes = {}
    ignorees = []

    for col in train.select_dtypes(include="number").columns:
        if col in colonnes_exclues:
            continue

        q1 = train[col].quantile(0.25)
        q3 = train[col].quantile(0.75)
        iqr = q3 - q1

        if iqr == 0:
            ignorees.append(col)          # bornes nulles : rien à faire
        else:
            bornes[col] = (q1 - facteur * iqr, q3 + facteur * iqr)

    print(f"IQR : {len(bornes)} variable(s) avec bornes ; {len(ignorees)} ignorée(s) (IQR = 0) : {ignorees}")
    return bornes


def appliquer_bornes_iqr(df, bornes):
    """Ramène à la borne les valeurs hors bornes. Les NaN restent des NaN."""
    df = df.copy()

    for col, (bas, haut) in bornes.items():
        if col not in df.columns:
            continue
        n_hors_bornes = int(((df[col] < bas) | (df[col] > haut)).sum())
        if n_hors_bornes > 0:
            print(f"  {col} : {n_hors_bornes} valeur(s) ramenée(s) dans [{bas:.2f} ; {haut:.2f}]")
        df[col] = df[col].clip(bas, haut) # La méthode clip() ramène les valeurs en dehors de l'intervalle vers la borne la plus proche.

    return df


# data preprocessing 


def fit_preprocessor(df, target_col='SalePrice'):

    df = df.copy()

    # ==========================================================
    # 0. AJOUT : retirer GrLivArea > 4000 (dans le TRAIN seulement)
    # ==========================================================

    df = retirer_grandes_maisons(df)

    # ==========================================================
    # 1. Séparer immédiatement X et y de TRAIN
    # ==========================================================

    y_train = df[target_col].copy()

    X_train = df.drop(
        columns=[target_col]
    ).copy()
    # ==========================================================
    # 2. Supprimer Id
    # ==========================================================

    if 'Id' in X_train.columns:
        X_train = X_train.drop(columns=['Id'])

    # ==========================================================
    # 3. Pourcentage de valeurs manquantes
    # ==========================================================

    nan_percentage = X_train.isna().mean() * 100

    print("Pourcentage de valeurs manquantes :")
    print(
        nan_percentage[nan_percentage > 0]
        .sort_values(ascending=False)
    )

    # ==========================================================
    # 4. Colonnes avec au moins 90 % de valeurs manquantes
    # ==========================================================

    colonnes_supprimees = nan_percentage[
        nan_percentage >= 90
    ].index.tolist()

    X_train = X_train.drop(
        columns=colonnes_supprimees
    )

    print(
        "Nombre de colonnes supprimées :",
        len(colonnes_supprimees)
    )

    # ==========================================================
    # 6. Identifier les variables numériques
    #    et catégorielles
    # ==========================================================

    variables_numeriques = X_train.select_dtypes(
        include="number"
    ).columns.tolist()

    # "toutes les colonnes non numériques" (avec pandas 3,
    # include=["object", "category"] ne détecte plus les colonnes de texte)
    variables_categorielles = [
        col for col in X_train.columns
        if col not in variables_numeriques
    ]

    # ==========================================================
    # 6b. AJOUT : IQR x 3 sur les variables numériques
    #     (avant l'imputation : les NaN sont ignorés dans le calcul des quartiles)
    #     Bornes apprises sur le TRAIN ; IQR = 0 -> on ne fait rien.
    # ==========================================================

    bornes_iqr = apprendre_bornes_iqr(X_train)

    colonnes_outliers = list(bornes_iqr.keys())

    X_train = appliquer_bornes_iqr(X_train, bornes_iqr)

    # ==========================================================
    # 7. Imputation initiale des variables numériques
    #    avec la médiane du TRAIN
    # ==========================================================

    medianes = {}

    for col in variables_numeriques:

        medianes[col] = X_train[col].median()

        X_train[col] = X_train[col].fillna(
            medianes[col]
        )

    # ==========================================================
    # 8. Imputation initiale des variables catégorielles
    #    avec le mode du TRAIN
    # ==========================================================

    modes = {}

    for col in variables_categorielles:

        mode = X_train[col].mode()

        if not mode.empty:

            modes[col] = mode.iloc[0]

            X_train[col] = X_train[col].fillna(
                modes[col]
            )

    # ==========================================================
    # 11. Standardisation
    # ==========================================================

    colonnes_standardiser = [
        col
        for col in variables_numeriques
    ]

    scaler = StandardScaler()

    X_train[colonnes_standardiser] = (
        scaler.fit_transform(
            X_train[colonnes_standardiser]
        )
    )



    # ==========================================================
    # 12. One-Hot Encoding
    # ==========================================================

    preprocessor = ColumnTransformer(
        transformers=[
            (
                'onehot',
                OneHotEncoder(
                    sparse_output=False,
                    handle_unknown='ignore'
                ),
                variables_categorielles
            )
        ],
        remainder='passthrough'
    )

    # Apprentissage + transformation sur TRAIN
    X_array = preprocessor.fit_transform(X_train)

    # ==========================================================
    # 13. Récupérer les noms des colonnes finales
    # ==========================================================

    cat_encoded_names = (
        preprocessor
        .named_transformers_['onehot']
        .get_feature_names_out(
            variables_categorielles
        )
    )

    # Colonnes numériques/celles laissées telles quelles
    autres_colonnes = [
        col
        for col in X_train.columns
        if col not in variables_categorielles
    ]

    final_columns = (
        list(cat_encoded_names)
        +
        autres_colonnes
    )

    # ==========================================================
    # 14. Reconstruire le DataFrame final
    # ==========================================================

    X_processed = pd.DataFrame(
        X_array,
        columns=final_columns,
        index=X_train.index
    )

    # ==========================================================
    # 15. Stocker toutes les transformations apprises
    # ==========================================================

    transformations = {

        # Colonnes supprimées
        'colonnes_supprimees':
            colonnes_supprimees,

        # Types de variables
        'variables_numeriques':
            variables_numeriques,

        'variables_categorielles':
            variables_categorielles,

        # Imputation
        'medianes':
            medianes,

        'modes':
            modes,

        # Outliers
        'colonnes_outliers':
            colonnes_outliers,

        'bornes_iqr':
            bornes_iqr,


        # Standardisation
        'colonnes_standardiser':
            colonnes_standardiser,

        'scaler':
            scaler,

        # One-Hot Encoding
        'preprocessor':
            preprocessor,

        # Colonnes finales
        'final_columns':
            final_columns
    }

    return X_processed, y_train, transformations

preprocessin pour le test

In [ ]:

# ==========================================================
# AJOUT : appliquer sur le TEST ce qui a été appris sur le TRAIN
#         (mêmes étapes que fit_preprocessor, mais rien n'est recalculé)
# ==========================================================
 
def transform_data(df, transformations, target_col='SalePrice'):
 
    X = df.copy()
 
    # ==========================================================
    # 1. Retirer la cible si elle est présente, et Id
    #    (aucune ligne n'est retirée : GrLivArea > 4000 ne concerne que le train)
    # ==========================================================
 
    X = X.drop(columns=[target_col], errors='ignore')
    y = df[target_col] if target_col in df.columns else None
 
    if 'Id' in X.columns:
        X = X.drop(columns=['Id'])
 
    # ==========================================================
    # 2. Supprimer les mêmes colonnes que sur le train
    # ==========================================================
 
    X = X.drop(
        columns=transformations['colonnes_supprimees'],
        errors='ignore'
    )
 
    # ==========================================================
    # 3. Mêmes colonnes, dans le même ordre que sur le train
    #    (une colonne absente du test provoque une erreur claire)
    # ==========================================================
 
    colonnes_train = list(transformations['preprocessor'].feature_names_in_)
    X = X[colonnes_train]
 
    # ==========================================================
    # 6b. IQR : bornes du TRAIN (avant l'imputation, comme dans fit_preprocessor)
    # ==========================================================
 
    X = appliquer_bornes_iqr(X, transformations['bornes_iqr'])
 
    # ==========================================================
    # 7. Imputation des variables numériques : médianes du TRAIN
    # ==========================================================
 
    for col, mediane in transformations['medianes'].items():
        X[col] = X[col].fillna(mediane)
 
    # ==========================================================
    # 8. Imputation des variables catégorielles : modes du TRAIN
    # ==========================================================
 
    for col, mode in transformations['modes'].items():
        X[col] = X[col].fillna(mode)
 
    # ==========================================================
    # 11. Standardisation : moyenne et écart-type du TRAIN
    # ==========================================================
 
    colonnes_standardiser = transformations['colonnes_standardiser']
 
    X[colonnes_standardiser] = (
        transformations['scaler'].transform(
            X[colonnes_standardiser]
        )
    )
 
    # ==========================================================
    # 12. One-Hot Encoding : catégories du TRAIN
    #     (une catégorie inconnue donne des 0 partout, sans erreur)
    # ==========================================================
 
    X_array = transformations['preprocessor'].transform(X)
 
    # ==========================================================
    # 14. Reconstruire le DataFrame final
    # ==========================================================
 
    X_processed = pd.DataFrame(
        X_array,
        columns=transformations['final_columns'],
        index=X.index
    )
 
    return X_processed, y

In [ ]:
X_train, y_train, transformations = fit_preprocessor(data_train)
X_test, y_test = transform_data(data_test, transformations)

GrLivArea > 4000 : 3 maison(s) retirée(s).
Pourcentage de valeurs manquantes :
MasVnrType     58.540773
LotFrontage    18.626609
Garage_Age      5.493562
MasVnrArea      0.515021
Electrical      0.085837
dtype: float64
Nombre de colonnes supprimées : 0
IQR : 28 variable(s) avec bornes ; 11 ignorée(s) (IQR = 0) : ['BsmtFinSF2', 'LowQualFinSF', 'BsmtHalfBath', 'KitchenAbvGr', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'IsNewHouse', 'Has_Garage']
  LotFrontage : 18 valeur(s) ramenée(s) dans [3.19 ; 5.29]
  LotArea : 30 valeur(s) ramenée(s) dans [7.64 ; 10.66]
  OverallCond : 1 valeur(s) ramenée(s) dans [2.00 ; 9.00]
  TotalBsmtSF : 29 valeur(s) ramenée(s) dans [5.22 ; 8.63]
  BedroomAbvGr : 1 valeur(s) ramenée(s) dans [-1.00 ; 6.00]
  TotRmsAbvGrd : 1 valeur(s) ramenée(s) dans [-1.00 ; 13.00]
  GarageArea : 1 valeur(s) ramenée(s) dans [-376.00 ; 1290.00]
  LotFrontage : 6 valeur(s) ramenée(s) dans [3.19 ; 5.29]
  LotArea : 9 valeur(s) ramenée(s) dans [7.64 ; 10.66

Features eng

In [ ]:
def fit_feature_engineering(x, y):

    x = x.copy()

    # --------------------------------------------------
    # 1. Variables numériques
    # --------------------------------------------------

    num_data = x.select_dtypes(
        include=np.number
    )

    # --------------------------------------------------
    # 2. Corrélation avec la cible
    # --------------------------------------------------

    corr = num_data.corrwith(y)

    corr_abs = corr.abs().sort_values(
        ascending=False
    )

    print("Corrélation avec la variable cible :")
    print(corr_abs)

    # --------------------------------------------------
    # 3. Sélection selon la corrélation avec y
    # --------------------------------------------------

    threshold = 0.3

    selected_features = corr_abs[
        corr_abs > threshold
    ].index.tolist()

    selected_data = num_data[
        selected_features
    ]

    # --------------------------------------------------
    # 4. Corrélation entre les features sélectionnées
    # --------------------------------------------------

    corr_matrix = selected_data.corr().abs()

    upper = corr_matrix.where(
        np.triu(
            np.ones(corr_matrix.shape),
            k=1
        ).astype(bool)
    )

    # --------------------------------------------------
    # 5. Features à supprimer si corrélation > 0.9
    # --------------------------------------------------

    to_drop = [
        column
        for column in upper.columns
        if any(upper[column] > 0.9)
    ]

    # --------------------------------------------------
    # 6. Features finales
    # --------------------------------------------------

    final_features = [
        column
        for column in selected_features
        if column not in to_drop
    ]

    # --------------------------------------------------
    # 7. Dataset final TRAIN
    # --------------------------------------------------

    x_selected = x[
        final_features
    ].copy()

    # --------------------------------------------------
    # 8. Stocker les décisions prises sur TRAIN
    # --------------------------------------------------

    feature_info = {
        'threshold': threshold,
        'selected_features': selected_features,
        'to_drop': to_drop,
        'final_features': final_features
    }

    print("\nVariables supprimées :", to_drop)

    print(
        "Nombre de variables finales :",
        len(final_features)
    )

    print("\nVariables finales :")
    print(final_features)

    return x_selected, y ,feature_info

In [ ]:
def transform_feature_engineering(x, feature_info ):
    x = x.copy()

    manquantes = [
        c for c in feature_info['final_features']
        if c not in x.columns
    ]

    if manquantes:
        raise KeyError(f"Colonnes absentes du jeu à transformer : {manquantes}")

    return x[feature_info['final_features']].copy()

receperer les donneées 

In [ ]:

X_train_selected, y_train_selected, feature_info = fit_feature_engineering(X_train, y_train)
X_test_selected   = transform_feature_engineering(X_test, feature_info=feature_info) 

Corrélation avec la variable cible :
OverallQual          0.811864
GrLivArea            0.729668
GarageCars           0.684663
GarageArea           0.661653
1stFlrSF             0.600386
                       ...   
Foundation_Stone     0.002975
MSSubClass_80        0.002699
RoofStyle_Mansard    0.002468
MiscFeature_Gar2     0.001305
MoSold_cos           0.000756
Length: 316, dtype: float64

Variables supprimées : ['ExterQual_Gd', 'Fireplaces', 'CentralAir_N', 'GarageCond_None', 'GarageFinish_None', 'GarageType_None', 'GarageQual_None', 'Exterior1st_VinylSd', 'SaleType_New']
Nombre de variables finales : 48

Variables finales :
['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', '1stFlrSF', 'TotalBsmtSF', 'ExterQual_TA', 'FullBath', 'House_Age', 'Remod_Age', 'KitchenQual_TA', 'TotRmsAbvGrd', 'Foundation_PConc', 'FireplaceQu_None', 'HeatingQC_Ex', 'Garage_Age', 'BsmtQual_TA', 'OpenPorchSF', 'BsmtQual_Ex', 'BsmtFinType1_GLQ', 'GarageFinish_Unf', 'KitchenQual_Ex', 'LotArea', 'Garage

Prediction

In [ ]:
# predict par regression lineaire
# chargment de model 
model = LinearRegression()

# Entrainment 
model.fit(X_train_selected, y_train)

# prediction
y_pred = model.predict(X_train_selected)


In [ ]:
# evaluation 
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred))
r2_train = r2_score(y_train, y_pred)
print("===== TRAIN =====")
print("RMSE:", rmse_train)
print("R²  :", r2_train)

===== TRAIN =====
RMSE: 0.12282712232837083
R²  : 0.9000340875130495


In [ ]:
# predict test par regression lineaire
y_test_pred = model.predict(X_test_selected)

# evaluation de test 
rmse_test = np.sqrt(mean_squared_error(y_test , y_test_pred))  
r2_test = r2_score(y_test , y_test_pred)
print("===== TEST =====")
print("RMSE:", rmse_test)
print("R²  :", r2_test)

===== TEST =====
RMSE: 0.14790281735515368
R²  : 0.8827762312016509


In [ ]:
# prediction avec rondom forest 
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train_selected, y_train)

y_pred_rf = model_rf.predict(X_train_selected)

# evaluation de train
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_pred_rf))
r2_train_rf = r2_score(y_train, y_pred_rf)
print("===== TRAIN RANDOM FOREST =====")
print("RMSE:", rmse_train_rf)
print("R²  :", r2_train_rf)

NameError: name 'X_train_selected' is not defined

In [ ]:
# test prediction avec random forest 
y_test_pred_rf = model_rf.predict(X_test_selected)

# evaluation de test
rmse_test_rf = np.sqrt(mean_squared_error(y_test , y_test_pred_rf))
r2_test_rf = r2_score(y_test , y_test_pred_rf)
print("===== TEST RANDOM FOREST =====")
print("RMSE:", rmse_test_rf)
print("R²  :", r2_test_rf)

===== TEST RANDOM FOREST =====
RMSE: 0.15627452299092454
R²  : 0.8691302869102506


In [ ]:
#prediction avec XGBoost
model_xgb = XGBRegressor(n_estimators=100, random_state=42)
model_xgb.fit(X_train_selected, y_train)
y_pred_xgb = model_xgb.predict(X_train_selected)

# evaluation de train
rmse_train_xgb = np.sqrt(mean_squared_error(y_train, y_pred_xgb))
r2_train_xgb = r2_score(y_train, y_pred_xgb)
print("===== TRAIN XGBOOST =====")
print("RMSE:", rmse_train_xgb)
print("R²  :", r2_train_xgb)

===== TRAIN XGBOOST =====
RMSE: 0.007842624042735988
R²  : 0.9995924450489762


In [ ]:
#evaluation 

y_test_pred_xgb = model_xgb.predict(X_test_selected)

# evaluation de test
rmse_test_xgb = np.sqrt(mean_squared_error(y_test , y_test_pred_xgb))
r2_test_xgb = r2_score(y_test , y_test_pred_xgb)
print("===== TEST XGBOOST =====")
print("RMSE:", rmse_test_xgb)
print("R²  :", r2_test_xgb)

===== TEST XGBOOST =====
RMSE: 0.16579813765434803
R²  : 0.8526934418091073


In [ ]:
# prediction with SVR 
model_svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=.1)
model_svr.fit(X_train_selected, y_train)
y_pred_svr = model_svr.predict(X_train_selected)

In [ ]:
# evaluation de train
mse_svr = mean_squared_error(y_train, y_pred_svr)
r2_svr = r2_score( y_train, y_pred_svr)
print('=======train=======')
print("RMSE", mse_svr)
print('R²', r2_svr)

# evaluation de test 
y_test_pred_svr = model_svr.predict(X_test_selected)

mse_svr_test = mean_squared_error(y_test, y_test_pred_svr)
r2_svr_test = r2_score( y_test, y_test_pred_svr)
print('=======train=======')
print("RMSE", mse_svr_test)
print('R²', r2_svr_test)


=======train=======
RMSE 0.00590982288450217
R² 0.9608404361667454
=======train=======
RMSE 0.033858100363486264
R² 0.8185632013447444


In [ ]:
# ==========================================================
# SOUMISSION KAGGLE
# ==========================================================

# 1. Lire test.csv avec EXACTEMENT les mêmes options que train.csv
test_raw = pd.read_csv("data/raw/test.csv", keep_default_na=False, na_values=["NA", ""])

# 2. Sauvegarder les Id avant toute transformation
ids = test_raw["Id"].copy()

# 3. Préparation préliminaire : la même fonction que pour le train
#    (équipements absents -> "None", âges, log1p, MSSubClass en texte)
test_kaggle = data_filter(test_raw)

# 4. Ce qui a été appris sur le train (rien n'est recalculé)
X_kaggle , _   = transform_data(test_kaggle, transformations)
X_kaggle_sel = transform_feature_engineering(X_kaggle, feature_info)

# 5. Prédiction (en log), puis retour en dollars
pred_log     = model_xgb.predict(X_kaggle_sel)
pred_dollars = np.expm1(pred_log)

# 6. Fichier de soumission
submission = pd.DataFrame({"Id": ids, "SalePrice": pred_dollars})
submission.to_csv("data/raw/submission.csv", index=False)

# 7. Vérifications avant d'envoyer
print(submission.shape)                                   # 1459 lignes pour Kaggle
print(submission["SalePrice"].describe())                 # de l'ordre de 50 000 à 500 000
assert submission["SalePrice"].notna().all()
assert len(submission) == len(test_raw)
assert submission["Id"].is_unique

  LotFrontage : 29 valeur(s) ramenée(s) dans [3.19 ; 5.29]
  LotArea : 39 valeur(s) ramenée(s) dans [7.64 ; 10.66]
  OverallCond : 6 valeur(s) ramenée(s) dans [2.00 ; 9.00]
  TotalBsmtSF : 43 valeur(s) ramenée(s) dans [5.22 ; 8.63]
  TotRmsAbvGrd : 1 valeur(s) ramenée(s) dans [-1.00 ; 13.00]
  GarageArea : 3 valeur(s) ramenée(s) dans [-376.00 ; 1290.00]
  Garage_Age : 1 valeur(s) ramenée(s) dans [-117.00 ; 170.00]
(1459, 2)
count      1459.000000
mean     177402.125000
std       76358.906250
min       43199.769531
25%      127047.730469
50%      156542.781250
75%      205459.570312
max      539715.937500
Name: SalePrice, dtype: float64


In [ ]:
resultas = pd.read_csv('data/raw/submission.csv')
resultas.head()

,Id,SalePrice
0,1461,122972.445
1,1462,150019.980
2,1463,186691.920
3,1464,192347.060
4,1465,175273.420


In [ ]:
# ==========================================================
# SOUMISSION KAGGLE
# ==========================================================

# 2. Sauvegarder les Id avant toute transformation
ids = test_raw["Id"].copy()

# 3. Préparation préliminaire : la même fonction que pour le train
#    (équipements absents -> "None", âges, log1p, MSSubClass en texte)
test_kaggle = data_filter(test_raw)

# 4. Ce qui a été appris sur le train (rien n'est recalculé)
X_kaggle , _   = transform_data(test_kaggle, transformations)
X_kaggle_sel = transform_feature_engineering(X_kaggle, feature_info)

# 5. Prédiction (en log), puis retour en dollars
pred_log     = model_rf.predict(X_kaggle_sel)
pred_dollars = np.expm1(pred_log)

# 6. Fichier de soumission
submission = pd.DataFrame({"Id": ids, "SalePrice": pred_dollars})
submission.to_csv("data/raw/submission_rf.csv", index=False)

# 7. Vérifications avant d'envoyer
print(submission.shape)                                   # 1459 lignes pour Kaggle
print(submission["SalePrice"].describe())                 # de l'ordre de 50 000 à 500 000
assert submission["SalePrice"].notna().all()
assert len(submission) == len(test_raw)
assert submission["Id"].is_unique

  LotFrontage : 29 valeur(s) ramenée(s) dans [3.19 ; 5.29]
  LotArea : 39 valeur(s) ramenée(s) dans [7.64 ; 10.66]
  OverallCond : 6 valeur(s) ramenée(s) dans [2.00 ; 9.00]
  TotalBsmtSF : 43 valeur(s) ramenée(s) dans [5.22 ; 8.63]
  TotRmsAbvGrd : 1 valeur(s) ramenée(s) dans [-1.00 ; 13.00]
  GarageArea : 3 valeur(s) ramenée(s) dans [-376.00 ; 1290.00]
  Garage_Age : 1 valeur(s) ramenée(s) dans [-117.00 ; 170.00]
(1459, 2)
count      1459.000000
mean     176620.510672
std       71332.706035
min       52301.196788
25%      127655.531957
50%      157296.488488
75%      205873.039681
max      516769.962952
Name: SalePrice, dtype: float64
